In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

def eisenstein_Ep_square(Z: np.ndarray, p: int, M: int = 12) -> np.ndarray:
    """
    Ep(z) = sum_{(m,n)!=(0,0)} 1 / (z + m + i n)^p
    Obcięcie: |m|,|n| <= M
    """
    Z = np.asarray(Z, dtype=np.complex128)
    Ep = np.zeros_like(Z, dtype=np.complex128)

    shifts = range(-M, M + 1)

    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        for m in shifts:
            for n in shifts:
                if m == 0 and n == 0:
                    continue
                Ep += 1.0 / (Z + (m + 1j * n))**p

    return Ep

In [5]:
def compute_epp_mod_from_csv(csv_path: Path, p: int, M: int = 12) -> dict:
    df = pd.read_csv(csv_path)

    a = df["a_re"].to_numpy() + 1j * df["a_im"].to_numpy()
    r = df["r_norm"].to_numpy()
    N = len(a)
    if N < 2:
        raise ValueError("Za mało obiektów (N < 2).")

    # wagi
    w = np.pi * r**2
    f = w.sum()

    # macierz różnic
    Z = a[:, None] - a[None, :]
    np.fill_diagonal(Z, np.nan + 1j*np.nan)

    Ep = eisenstein_Ep_square(Z, p=p, M=M)
    Ep = np.where(np.isfinite(Ep.real), Ep, 0.0 + 0.0j)

    # wyzeruj przekątną (konsekwentnie)
    np.fill_diagonal(Ep, 0.0 + 0.0j)

    # u_j = sum_i w_i Ep(i,j)  => (w^T Ep) jako wektor po kolumnach
    u = (w[:, None] * Ep).sum(axis=0)                 # (N,)

    # v_j = sum_k w_k conj(Ep(j,k))
    v = (w[None, :] * np.conjugate(Ep)).sum(axis=1)   # (N,)

    epp_mod = ( (w**(p-1)) * u * v ).sum() / (f**(p+1))

    return {
        "N": int(N),
        "M": int(M),
        "p": int(p),
        "f": float(f),
        "epp_mod_re": float(epp_mod.real),
        "epp_mod_im": float(epp_mod.imag),
        "epp_mod_abs": float(np.abs(epp_mod)),
    }


In [6]:
res33 = compute_epp_mod_from_csv(args.csv, p=3, M=args.M)
res88 = compute_epp_mod_from_csv(args.csv, p=8, M=args.M)
print("\n=== epp_mod (p=3) ===", res33)
print("\n=== epp_mod (p=8) ===", res88)

NameError: name 'args' is not defined

In [1]:
import pandas as pd

In [2]:
pd.read_csv('haadf_disks_bright.csv')

,a_re,a_im,r_norm,cx_px,cy_px,r_px
0,-0.027475,-0.469921,0.048660,967.731859,61.962949,99.655750
1,0.140846,-0.485694,0.022214,1312.451707,29.470471,45.493416
2,0.231048,-0.483110,0.018427,1497.186634,34.793473,37.737494
3,0.457435,-0.449265,0.051450,1960.827843,104.514893,105.370618
4,-0.040956,-0.491783,0.001764,940.121951,16.926829,3.612576
...,...,...,...,...,...,...
84,0.209810,0.418262,0.001785,1453.690476,1891.619048,3.656366
85,0.205339,0.424072,0.002613,1444.533333,1903.588889,5.352372
86,0.241811,0.463076,0.042886,1519.229173,1983.937570,87.830747
87,0.469580,0.477471,0.020634,1985.699465,2013.590196,42.257762


In [7]:
pd.set_option('display.float_format', '{:.8f}'.format)
pd.read_csv('features_haadf.csv').T

,0
N,89.00000000
M,12.00000000
f,0.24431419
e2_re,8.59813082
e2_im,14.99696796
e2_abs,17.28689971
e2_mod_re,-1.55060026
e2_mod_im,-0.78155271
e2_mod_abs,1.73642904
e33_mod_re,-13.42499568
